Cell 1: Install Libraries

In [ ]:
!pip install openai numpy pandas scipy

Cell 2: Import & Config

In [ ]:
import random
import time
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.stats import norm
import os
import glob
from google.colab import userdata

# Retrieve the SiliconFlow API key from Google Colab secrets
API_KEY_ENV = userdata.get('siliconflow_api_key')
if not API_KEY_ENV:
    raise ValueError("SILICONFLOW_API_KEY not found in Google Colab secrets.")

BASE_URL = "https://api.siliconflow.cn/v1"
MODEL_NAMES_TO_TEST = [
    "Qwen/Qwen3-32B",
    "deepseek-ai/DeepSeek-V3"
]
N_BACK_VALUES = [1, 2]
NUM_BLOCKS_PER_N = 10
TRIALS_PER_BLOCK = 30
FIXED_DATASET_PATH_1BACK = "1back/"
FIXED_DATASET_PATH_2BACK = "2back/"

# Initialize the OpenAI client with the SiliconFlow API key and base URL
client = OpenAI(api_key=API_KEY_ENV, base_url=BASE_URL)

Cell 3: Helper Functions

In [ ]:
def load_sequence_and_targets_from_file(file_path):
    with open(file_path, 'r') as f:
        content = f.read().strip()
    sequence_part = content[:TRIALS_PER_BLOCK]
    targets_part = content[TRIALS_PER_BLOCK:]
    sequence = list(sequence_part.upper())
    targets = list(targets_part.lower())
    return sequence, targets

def generate_block(n_back, seed=None):
    if seed: random.seed(seed)
    initial_len = n_back + TRIALS_PER_BLOCK
    seq = random.choices(list('ABCDEFGHIJKLMNOPQRSTUVWXYZ'), k=initial_len)
    targets = []
    for i in range(n_back, initial_len):
        if seq[i] == seq[i - n_back]: targets.append('m')
        else: targets.append('-')
    return seq[n_back:n_back + TRIALS_PER_BLOCK], targets[:TRIALS_PER_BLOCK]

def get_model_response(client, prompt, model_name, temperature=0.7):
    try:
        completion = client.chat.completions.create(
            model=model_name, messages=[{"role": "user", "content": prompt}],
            max_tokens=10, temperature=temperature
        )
        response_text = completion.choices[0].message.content.strip().lower()
        for char in response_text:
            if char == 'm': return 'm'
            elif char in ['-', 'n']: return '-'
        return '-'
    except Exception as e:
        print(f"API call failed: {e}")
        return '-'

Cell 4: Main Experiment Function

In [ ]:
def run_single_condition(n_back, num_blocks, client, model_name, temp_value, use_fixed_dataset=False, condition_label=""):
    print(f"Starting {condition_label}...")
    all_hits, all_misses, all_false_alarms, all_correct_rejections = 0, 0, 0, 0

    if use_fixed_dataset:
        path = FIXED_DATASET_PATH_1BACK if n_back == 1 else FIXED_DATASET_PATH_2BACK
        file_paths = sorted(glob.glob(os.path.join(path, "*.txt")))[:num_blocks]
        for block_num, file_path in enumerate(file_paths):
            print(f"  Loading Fixed Dataset Block {block_num+1} from {file_path}...")
            sequence, targets = load_sequence_and_targets_from_file(file_path)
            print(f"    Sequence: {sequence}")
            print(f"    Targets:  {targets}")

            model_responses = []
            for i, letter in enumerate(sequence):
                context_str = ", ".join(sequence[:i+1])
                prompt = f"N-back task, N is {n_back}. Respond 'm' if current letter matches the one {n_back} steps back, otherwise '-'. Sequence so far: {context_str}. Your response?"
                resp = get_model_response(client, prompt, model_name, temperature=temp_value)
                model_responses.append(resp)
            print(f"    Responses: {model_responses}")

            block_hits, block_misses, block_false_alarms, block_correct_rejections = 0, 0, 0, 0
            for t, r in zip(targets, model_responses):
                if t == 'm':
                    if r == 'm':
                        block_hits += 1
                        all_hits += 1
                    else:
                        block_misses += 1
                        all_misses += 1
                else: # t == '-'
                    if r == '-':
                        block_correct_rejections += 1
                        all_correct_rejections += 1
                    else:
                        block_false_alarms += 1
                        all_false_alarms += 1
            print(f"    Block {block_num+1} Stats: H={block_hits}, M={block_misses}, FA={block_false_alarms}, CR={block_correct_rejections}")
    else:
        for block_num in range(num_blocks):
            print(f"  Running Dynamic Generation Block {block_num+1}...")
            sequence, targets = generate_block(n_back, seed=block_num)
            print(f"    Sequence: {sequence}")
            print(f"    Targets:  {targets}")

            model_responses = []
            for i, letter in enumerate(sequence):
                context_str = ", ".join(sequence[:i+1])
                prompt = f"N-back task, N is {n_back}. Respond 'm' if current letter matches the one {n_back} steps back, otherwise '-'. Sequence so far: {context_str}. Your response?"
                resp = get_model_response(client, prompt, model_name, temperature=temp_value)
                model_responses.append(resp)
            print(f"    Responses: {model_responses}")

            block_hits, block_misses, block_false_alarms, block_correct_rejections = 0, 0, 0, 0
            for t, r in zip(targets, model_responses):
                if t == 'm':
                    if r == 'm':
                        block_hits += 1
                        all_hits += 1
                    else:
                        block_misses += 1
                        all_misses += 1
                else: # t == '-'
                    if r == '-':
                        block_correct_rejections += 1
                        all_correct_rejections += 1
                    else:
                        block_false_alarms += 1
                        all_false_alarms += 1
            print(f"    Block {block_num+1} Stats: H={block_hits}, M={block_misses}, FA={block_false_alarms}, CR={block_correct_rejections}")

    total_signal = all_hits + all_misses
    total_noise = all_false_alarms + all_correct_rejections
    total_trials = total_signal + total_noise
    hit_rate = all_hits / total_signal if total_signal > 0 else 0.0
    fa_rate = all_false_alarms / total_noise if total_noise > 0 else 0.0
    accuracy = (all_hits + all_correct_rejections) / total_trials if total_trials > 0 else 0.0
    epsilon = 1e-9
    corrected_hr = np.clip(hit_rate, epsilon, 1 - epsilon)
    corrected_far = np.clip(fa_rate, epsilon, 1 - epsilon)
    d_prime = norm.ppf(corrected_hr) - norm.ppf(corrected_far)
    print(f"Summary for {condition_label}: Acc={accuracy:.4f}, d'={d_prime:.4f}")
    return {'accuracy': accuracy, 'd_prime': d_prime}

Cell 5: Run Experiments & Compare

In [ ]:
# --- QUICK TEST CONFIGURATION ---
QUICK_NUM_BLOCKS_PER_N = 2 # Reduce blocks for quick test
QUICK_MODELS_TO_TEST = ["Qwen/Qwen3-32B"] # Test only one model first
QUICK_TEMPS = [0.7]
# Loop through boolean values for use_fixed_dataset
USE_FIXED_DATASET_VALUES = [False, True] # False for dynamic, True for fixed

# Store quick test results
quick_results = {}

for model_name in QUICK_MODELS_TO_TEST:
    print(f"\n\n{'='*20} Quick Test: {model_name} {'='*20}")
    model_results = {}

    for n_val in N_BACK_VALUES:
        n_results = {}

        for temp_val in QUICK_TEMPS:
            temp_results = {}

            for use_fixed in USE_FIXED_DATASET_VALUES:
                dataset_type = "fixed" if use_fixed else "dynamic"
                condition_label = f"Quick-{model_name}_{dataset_type}_T{temp_val}_{n_val}back"

                # Determine number of blocks for quick test
                num_blocks_to_run = min(QUICK_NUM_BLOCKS_PER_N, 50) if use_fixed else QUICK_NUM_BLOCKS_PER_N

                res = run_single_condition(
                    n_back=n_val,
                    num_blocks=num_blocks_to_run,
                    client=client,
                    model_name=model_name,
                    temp_value=temp_val,
                    use_fixed_dataset=use_fixed,
                    condition_label=condition_label
                )
                # Store result using the string key 'dynamic' or 'fixed'
                temp_results[dataset_type] = res

            n_results[temp_val] = temp_results

        model_results[n_val] = n_results

    quick_results[model_name] = model_results

# Print quick test summary
print("\n\n" + "="*60)
print("QUICK TEST RESULTS SUMMARY")
print("="*60)
for model_name, model_data in quick_results.items():
    print(f"\n--- Model: {model_name} (Quick Test) ---")
    for n_val, n_data in model_data.items():
        print(f"  {n_val}-Back:")
        for temp_val, temp_data in n_data.items():
            # Access results using the correct keys now
            dyn_res = temp_data['dynamic']
            fix_res = temp_data['fixed']
            print(f"    T={temp_val}: Dyn Acc: {dyn_res['accuracy']:.4f}, d': {dyn_res['d_prime']:.4f} | Fix Acc: {fix_res['accuracy']:.4f}, d': {fix_res['d_prime']:.4f}")



==================== Quick Test: Qwen/Qwen3-32B ====================
Starting Quick-Qwen/Qwen3-32B_dynamic_T0.7_1back...
  Running Dynamic Generation Block 1...
    Sequence: ['B', 'A', 'W', 'B', 'D', 'E', 'B', 'K', 'A', 'I', 'W', 'M', 'O', 'S', 'C', 'C', 'L', 'J', 'M', 'E', 'J', 'V', 'A', 'Y', 'K', 'Z', 'K', 'A', 'K', 'R']
    Targets:  ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', 'm', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-']


In [ ]:
all_results = {}
for model_name in MODEL_NAMES_TO_TEST:
    model_results = {}
    for n_val in N_BACK_VALUES:
        n_results = {}
        for temp_val in [0.7, 0.1]:
            temp_results = {}
            for use_fixed in [False, True]:
                dataset_type = "fixed" if use_fixed else "dynamic"
                condition_label = f"{model_name}_{dataset_type}_T{temp_val}_{n_val}back"
                num_blocks_to_run = min(NUM_BLOCKS_PER_N, 50) if use_fixed else NUM_BLOCKS_PER_N
                res = run_single_condition(
                    n_back=n_val, num_blocks=num_blocks_to_run, client=client,
                    model_name=model_name, temp_value=temp_val,
                    use_fixed_dataset=use_fixed, condition_label=condition_label
                )
                temp_results[dataset_type] = res
        n_results[temp_val] = temp_results
        model_results[n_val] = n_results
    all_results[model_name] = model_results

# Print summary
for model_name, model_data in all_results.items():
    print(f"\n--- Model: {model_name} ---")
    for n_val, n_data in model_data.items():
        print(f"  {n_val}-Back:")
        for temp_val, temp_data in n_data.items():
            dyn_res = temp_data['dynamic']
            fix_res = temp_data['fixed']
            print(f"    T={temp_val}: Dyn Acc: {dyn_res['accuracy']:.4f}, d': {dyn_res['d_prime']:.4f} | Fix Acc: {fix_res['accuracy']:.4f}, d': {fix_res['d_prime']:.4f}")

# Cross-model comparison on fixed dataset
for n_val in N_BACK_VALUES:
    print(f"\n--- {n_val}-Back (Fixed Dataset) ---")
    for temp_val in [0.7, 0.1]:
        print(f"  T={temp_val}:")
        for model_name in MODEL_NAMES_TO_TEST:
            res = all_results[model_name][n_val][temp_val]['fixed']
            print(f"    {model_name}: Acc={res['accuracy']:.4f}, d'={res['d_prime']:.4f}")

Starting Qwen/Qwen3-32B_dynamic_T0.7_1back...


KeyboardInterrupt: 

Available Models on SiliconFlow:
----------------------------------------
Model ID: Pro/MiniMaxAI/MiniMax-M2.5 | Owned by:
Model ID: Pro/zai-org/GLM-5 | Owned by:
Model ID: Pro/moonshotai/Kimi-K2.5 | Owned by:
Model ID: Pro/zai-org/GLM-4.7 | Owned by:
Model ID: deepseek-ai/DeepSeek-V3.2 | Owned by:
Model ID: Pro/deepseek-ai/DeepSeek-V3.2 | Owned by:
Model ID: deepseek-ai/DeepSeek-V3.1-Terminus | Owned by:
Model ID: Pro/deepseek-ai/DeepSeek-V3.1-Terminus | Owned by:
Model ID: deepseek-ai/DeepSeek-R1 | Owned by:
Model ID: Pro/deepseek-ai/DeepSeek-R1 | Owned by:
Model ID: PaddlePaddle/PaddleOCR-VL-1.5 | Owned by:
Model ID: deepseek-ai/DeepSeek-V3 | Owned by:
Model ID: Pro/deepseek-ai/DeepSeek-V3 | Owned by:
Model ID: Pro/MiniMaxAI/MiniMax-M2.1 | Owned by:
Model ID: stepfun-ai/Step-3.5-Flash | Owned by:
Model ID: zai-org/GLM-4.6V | Owned by:
Model ID: moonshotai/Kimi-K2-Thinking | Owned by:
Model ID: Pro/moonshotai/Kimi-K2-Thinking | Owned by:
Model ID: zai-org/GLM-4.6 | Owned by:
Model ID: Kwaipilot/KAT-Dev | Owned by:
Model ID: PaddlePaddle/PaddleOCR-VL | Owned by:
Model ID: Qwen/Qwen3-VL-32B-Instruct | Owned by:
Model ID: Qwen/Qwen3-VL-32B-Thinking | Owned by:
Model ID: Qwen/Qwen3-VL-8B-Instruct | Owned by:
Model ID: Qwen/Qwen3-VL-8B-Thinking | Owned by:
Model ID: Qwen/Qwen3-VL-30B-A3B-Instruct | Owned by:
Model ID: Qwen/Qwen3-VL-30B-A3B-Thinking | Owned by:
Model ID: Qwen/Qwen3-VL-235B-A22B-Instruct | Owned by:
Model ID: Qwen/Qwen3-VL-235B-A22B-Thinking | Owned by:
Model ID: Qwen/Qwen3-Omni-30B-A3B-Instruct | Owned by:
Model ID: Qwen/Qwen3-Omni-30B-A3B-Thinking | Owned by:
Model ID: Qwen/Qwen3-Omni-30B-A3B-Captioner | Owned by:
Model ID: deepseek-ai/DeepSeek-OCR | Owned by:
Model ID: moonshotai/Kimi-K2-Instruct-0905 | Owned by:
Model ID: Pro/moonshotai/Kimi-K2-Instruct-0905 | Owned by:
Model ID: Qwen/Qwen3-Next-80B-A3B-Instruct | Owned by:
Model ID: Qwen/Qwen3-Next-80B-A3B-Thinking | Owned by:
Model ID: inclusionAI/Ring-flash-2.0 | Owned by:
Model ID: inclusionAI/Ling-flash-2.0 | Owned by:
Model ID: inclusionAI/Ling-mini-2.0 | Owned by:
Model ID: Qwen/Qwen-Image-Edit-2509 | Owned by:
Model ID: Qwen/Qwen-Image-Edit | Owned by:
Model ID: Qwen/Qwen-Image | Owned by:
Model ID: tencent/Hunyuan-MT-7B | Owned by:
Model ID: ByteDance-Seed/Seed-OSS-36B-Instruct | Owned by:
Model ID: Wan-AI/Wan2.2-I2V-A14B | Owned by:
Model ID: Wan-AI/Wan2.2-T2V-A14B | Owned by:
Model ID: zai-org/GLM-4.5V | Owned by:
Model ID: zai-org/GLM-4.5-Air | Owned by:
Model ID: TeleAI/TeleSpeechASR | Owned by:
Model ID: Qwen/Qwen3-Coder-30B-A3B-Instruct | Owned by:
Model ID: Qwen/Qwen3-Coder-480B-A35B-Instruct | Owned by:
Model ID: Qwen/Qwen3-30B-A3B-Thinking-2507 | Owned by:
Model ID: Qwen/Qwen3-30B-A3B-Instruct-2507 | Owned by:
Model ID: Qwen/Qwen3-235B-A22B-Thinking-2507 | Owned by:
Model ID: Qwen/Qwen3-235B-A22B-Instruct-2507 | Owned by:
Model ID: THUDM/GLM-4.1V-9B-Thinking | Owned by:
Model ID: baidu/ERNIE-4.5-300B-A47B | Owned by:
Model ID: tencent/Hunyuan-A13B-Instruct | Owned by:
Model ID: deepseek-ai/DeepSeek-R1-0528-Qwen3-8B | Owned by:
Model ID: Qwen/Qwen3-32B | Owned by:
Model ID: Qwen/Qwen3-14B | Owned by:
Model ID: Qwen/Qwen3-8B | Owned by:
Model ID: Qwen/Qwen3-Reranker-8B | Owned by:
Model ID: Qwen/Qwen3-Embedding-8B | Owned by:
Model ID: Qwen/Qwen3-Reranker-4B | Owned by:
Model ID: Qwen/Qwen3-Embedding-4B | Owned by:
Model ID: Qwen/Qwen3-Reranker-0.6B | Owned by:
Model ID: Qwen/Qwen3-Embedding-0.6B | Owned by:
Model ID: ascend-tribe/pangu-pro-moe | Owned by:
Model ID: THUDM/GLM-Z1-32B-0414 | Owned by:
Model ID: THUDM/GLM-4-32B-0414 | Owned by:
Model ID: THUDM/GLM-Z1-9B-0414 | Owned by:
Model ID: THUDM/GLM-4-9B-0414 | Owned by:
Model ID: Qwen/Qwen2.5-VL-32B-Instruct | Owned by:
Model ID: Qwen/QwQ-32B | Owned by:
Model ID: Qwen/Qwen2.5-VL-72B-Instruct | Owned by:
Model ID: Pro/Qwen/Qwen2.5-VL-7B-Instruct | Owned by:
Model ID: deepseek-ai/DeepSeek-R1-Distill-Qwen-32B | Owned by:
Model ID: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B | Owned by:
Model ID: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B | Owned by:
Model ID: deepseek-ai/DeepSeek-V2.5 | Owned by:
Model ID: fnlp/MOSS-TTSD-v0.5 | Owned by:
Model ID: FunAudioLLM/CosyVoice2-0.5B | Owned by:
Model ID: FunAudioLLM/SenseVoiceSmall | Owned by:
Model ID: IndexTeam/IndexTTS-2 | Owned by:
Model ID: BAAI/bge-m3 | Owned by:
Model ID: BAAI/bge-reranker-v2-m3 | Owned by:
Model ID: netease-youdao/bce-embedding-base_v1 | Owned by:
Model ID: netease-youdao/bce-reranker-base_v1 | Owned by:
Model ID: Qwen/Qwen2.5-Coder-32B-Instruct | Owned by:
Model ID: Kwai-Kolors/Kolors | Owned by:
Model ID: Qwen/Qwen2-VL-72B-Instruct | Owned by:
Model ID: Qwen/Qwen2.5-72B-Instruct-128K | Owned by:
Model ID: Qwen/Qwen2.5-72B-Instruct | Owned by:
Model ID: deepseek-ai/deepseek-vl2 | Owned by:
Model ID: Qwen/Qwen2.5-32B-Instruct | Owned by:
Model ID: Qwen/Qwen2.5-14B-Instruct | Owned by:
Model ID: Qwen/Qwen2.5-7B-Instruct | Owned by:
Model ID: Qwen/Qwen2.5-Coder-7B-Instruct | Owned by:
Model ID: internlm/internlm2_5-7b-chat | Owned by:
Model ID: Qwen/Qwen2-7B-Instruct | Owned by:
Model ID: THUDM/glm-4-9b-chat | Owned by:
Model ID: BAAI/bge-large-en-v1.5 | Owned by:
Model ID: BAAI/bge-large-zh-v1.5 | Owned by:
Model ID: LoRA/Qwen/Qwen2.5-32B-Instruct | Owned by:
Model ID: LoRA/Qwen/Qwen2.5-14B-Instruct | Owned by:
Model ID: Pro/Qwen/Qwen2.5-Coder-7B-Instruct | Owned by:
Model ID: Pro/BAAI/bge-m3 | Owned by:
Model ID: Pro/Qwen/Qwen2.5-7B-Instruct | Owned by:
Model ID: Pro/BAAI/bge-reranker-v2-m3 | Owned by:
Model ID: LoRA/Qwen/Qwen2.5-72B-Instruct | Owned by:
Model ID: Pro/Qwen/Qwen2-7B-Instruct | Owned by:
Model ID: LoRA/Qwen/Qwen2.5-7B-Instruct | Owned by:
Model ID: Pro/THUDM/glm-4-9b-chat | Owned by:
Model ID: black-forest-labs/FLUX.1-schnell | Owned by:
Model ID: black-forest-labs/FLUX.1-dev | Owned by:
Model ID: Pro/black-forest-labs/FLUX.1-schnell | Owned by:
Model ID: fishaudio/fish-speech-1.4 | Owned by:
Model ID: RVC-Boss/GPT-SoVITS | Owned by:
Model ID: fishaudio/fish-speech-1.5 | Owned by:
Model ID: black-forest-labs/FLUX.1-pro | Owned by:
Model ID: SeedLLM/Seed-Rice-7B | Owned by:
